## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as dset
from torchvision import datasets, transforms

## Set Global Device

In [2]:
# GPU
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU State:', device)

GPU State: cpu


## Functions

In [3]:
def training_loop(model, loss, optimizer, loader, epochs, verbose=True, device=device):
    """
    Run training of a model given a loss function, optimizer and a set of training and validation data.
    """

    # Train
    for epoch in range(epochs):
        running_loss = 0.0

        for times, data in enumerate(loader):
            inputs, labels = data[0].to(device), data[1].to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Foward + backward + optimize
            outputs = model(inputs)
            loss_tensor = loss(outputs, labels)
            loss_tensor.backward()
            optimizer.step()

            # Print statistics
            running_loss += loss_tensor.item()
            if verbose:
                if times % 100 == 99 or times+1 == len(loader):
                    print('[%d/%d, %d/%d] loss: %.3f' % (epoch+1, epochs, times+1, len(loader), running_loss/2000))

In [4]:
def evaluate_model(model, loader, device=device):
    """
    Evaluate a model 'model' on all batches of a torch DataLoader 'data_loader'.

    Returns: the total number of correct classifications,
             the total number of images
             the list of the per class correct classification,
             the list of the per class total number of images.
    """

    # Test
    correct = 0
    total = 0

    with torch.no_grad():
        for data in loader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    class_correct = [0 for i in range(10)]
    class_total = [0 for i in range(10)]

    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            c = (predicted == labels).squeeze()
            for i in range(len(labels)):
                label = labels[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1

    return (correct, total, class_correct, class_total)


## Main program

In [5]:
# Transform
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,), (0.5,)),]
)

In [6]:
# Data
trainSet = datasets.MNIST(root='MNIST', download=True, train=True, transform=transform)
testSet = datasets.MNIST(root='MNIST', download=True, train=False, transform=transform)
trainLoader = dset.DataLoader(trainSet, batch_size=64, shuffle=True)
testLoader = dset.DataLoader(testSet, batch_size=64, shuffle=False)

In [7]:
# Model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3, 3), stride=(1, 1)),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), stride=(1, 1)),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False),
            nn.Flatten(start_dim=1, end_dim=-1),
            nn.Linear(in_features=800, out_features=10, bias=True),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, input):
        return self.main(input)


net = Net().to(device)
print(net)

Net(
  (main): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=800, out_features=10, bias=True)
    (8): LogSoftmax(dim=1)
  )
)


In [8]:
# Parameters
epochs = 4
lr = 0.002
loss = nn.NLLLoss()
optimizer = optim.SGD(net.parameters(), lr=0.002, momentum=0.9)

# Train
print('Training on %d images' % trainSet.data.shape[0])
training_loop(net, loss, optimizer, trainLoader, epochs)
print('Training Finished.\n')

# Test
correct, total, class_correct, class_total = evaluate_model(net, testLoader)
print('Accuracy of the network on the %d test images: %d %%' % (testSet.data.shape[0], (100*correct / total)))
for i in range(10):
    print('Accuracy of %d: %3f' % (i, (class_correct[i]/class_total[i])))

Training on 60000 images
[1/4, 100/938] loss: 0.103
[1/4, 200/938] loss: 0.142
[1/4, 300/938] loss: 0.162
[1/4, 400/938] loss: 0.178
[1/4, 500/938] loss: 0.191
[1/4, 600/938] loss: 0.203
[1/4, 700/938] loss: 0.213
[1/4, 800/938] loss: 0.223
[1/4, 900/938] loss: 0.232
[1/4, 938/938] loss: 0.235
[2/4, 100/938] loss: 0.008
[2/4, 200/938] loss: 0.016
[2/4, 300/938] loss: 0.023
[2/4, 400/938] loss: 0.029
[2/4, 500/938] loss: 0.037
[2/4, 600/938] loss: 0.043
[2/4, 700/938] loss: 0.049
[2/4, 800/938] loss: 0.055
[2/4, 900/938] loss: 0.060
[2/4, 938/938] loss: 0.062
[3/4, 100/938] loss: 0.006
[3/4, 200/938] loss: 0.011
[3/4, 300/938] loss: 0.016
[3/4, 400/938] loss: 0.020
[3/4, 500/938] loss: 0.025
[3/4, 600/938] loss: 0.031
[3/4, 700/938] loss: 0.035
[3/4, 800/938] loss: 0.040
[3/4, 900/938] loss: 0.045
[3/4, 938/938] loss: 0.046
[4/4, 100/938] loss: 0.004
[4/4, 200/938] loss: 0.009
[4/4, 300/938] loss: 0.013
[4/4, 400/938] loss: 0.017
[4/4, 500/938] loss: 0.021
[4/4, 600/938] loss: 0.025
[4/

In [9]:
# Count parameters
total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Detailed breakdown
print("\nParameter breakdown by layer:")
for name, param in net.named_parameters():
    print(f"{name}: {param.numel():,} parameters (shape: {list(param.shape)})")

Total parameters: 12,810
Trainable parameters: 12,810

Parameter breakdown by layer:
main.0.weight: 144 parameters (shape: [16, 1, 3, 3])
main.0.bias: 16 parameters (shape: [16])
main.3.weight: 4,608 parameters (shape: [32, 16, 3, 3])
main.3.bias: 32 parameters (shape: [32])
main.7.weight: 8,000 parameters (shape: [10, 800])
main.7.bias: 10 parameters (shape: [10])


## **Answers to Task 1.3**


### **1.3.1) Why must the Linear layer be preceded by `Flatten` and why does the Linear layer have 800 input features?**

Convolutional and pooling layers output a 3-dimensional tensor of shape $(C, H, W)$. A linear (fully connected) layer, however, expects a 1-dimensional feature vector. The `Flatten` operation reshapes the final convolutional feature map into a single vector so that all spatial and channel-wise features can be used as input to the linear classifier.

After the second max-pooling layer, the feature map has dimensions
$$
32 \text{ channels} \times 5 \times 5,
$$
resulting from the sequence of valid convolutions and two (2 \times 2) pooling operations applied to a (28 \times 28) input. The total number of elements is therefore
$$
32 \cdot 5 \cdot 5 = 800,
$$
which becomes the size of the flattened vector and thus the required number of input features to the linear layer.

### **1.3.2) Number of params**

**Layer-by-layer parameter count:**

**1. `Conv2d(1, 16, kernel_size=3x3)`**

- **Weights:** \($1 \times 16 \times 3 \times 3 = 144$\)  
- **Bias:** 16  
- **Total:** **160 parameters**

**2. `Conv2d(16, 32, kernel_size=3x3)`**

- **Weights:** \($16 \times 32 \times 3 \times 3 = 4,608$\)  
- **Bias:** 32  
- **Total:** **4,640 parameters**

**3. `Linear(800, 10)`**

- **Weights:** \($800 \times 10 = 8,000$\)  
- **Bias:** 10  
- **Total:** **8,010 parameters**

**Total parameters:**  
\[
160 + 4,640 + 8,010 = $\mathbf{12,810 \ parameters}$
\]

### **1.3.3) How well is it able to correctly classify each digit class 0, 1, ..., 9?**
**Overall Test Accuracy: 97%** (on 10,000 test images)
**Per-class accuracy:**
- **Digit 0**: 99.18%
- **Digit 1**: 99.47%
- **Digit 2**: 97.97%
- **Digit 3**: 97.92%
- **Digit 4**: 98.17%
- **Digit 5**: 98.65%
- **Digit 6**: 97.18%
- **Digit 7**: 96.60%
- **Digit 8**: 98.15%
- **Digit 9**: 94.65%

NB: Highest on digits 0 and 1 (~99.4%); lowest on digit 9 (~94.6%).